# 🛰️ EuroSAT Sentinel-2: Exploratory Data Analysis (EDA)

This notebook explores the **Sentinel-2 EuroSAT** dataset for Land Use and Land Cover (LULC) classification.

### Key Objectives:
1. Verify the modular data pipeline (`src.data.dataset`).
2. Inspect class distribution and partition sizes (Train 70%, Validation 15%, Test 15%).
3. Visualize sample satellite images across all 10 categories.
4. Check pixel value distributions and channel statistics.


In [ ]:
import sys
from pathlib import Path

# Añadir la raíz del proyecto para importar desde src
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import torch
import numpy as np
import matplotlib.pyplot as plt
from src.data.dataset import get_dataloaders


## 1. Carga del Pipeline de Datos

Cargamos los DataLoaders estructurados. Observa cómo todo el preprocesamiento, split reproducible y prevención de data leakage se realiza en **una sola línea**.


In [ ]:
train_loader, val_loader, test_loader, class_names = get_dataloaders(
    data_dir='data/raw',
    batch_size=32,
    num_workers=0,
    seed=42
)

print(f"Total de clases ({len(class_names)}):\n{class_names}\n")
print(f"Batches de Train: {len(train_loader)} (~{len(train_loader.dataset)} imágenes)")
print(f"Batches de Val:   {len(val_loader)} (~{len(val_loader.dataset)} imágenes)")
print(f"Batches de Test:  {len(test_loader)} (~{len(test_loader.dataset)} imágenes)")


## 2. Visualización de Muestras Satelitales por Clase

Para visualizar las imágenes correctamente, desnormalizamos los tensores restando la media y dividiendo por la desviación estándar de ImageNet:


In [ ]:
# Función para desnormalizar imágenes de ImageNet a RGB visible [0, 1]
def denormalize(tensor):
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = tensor.permute(1, 2, 0).cpu().numpy()
    img = std * img + mean
    return np.clip(img, 0, 1)

# Tomamos un lote del conjunto de validación (imágenes limpias sin augmentations)
images, labels = next(iter(val_loader))

plt.figure(figsize=(15, 8))
for i in range(min(12, len(images))):
    plt.subplot(3, 4, i + 1)
    plt.imshow(denormalize(images[i]))
    plt.title(f"{class_names[labels[i]]}", fontsize=11, fontweight='bold')
    plt.axis('off')

plt.suptitle("Muestras Satelitales Sentinel-2 EuroSAT (224x224)", fontsize=14, y=0.98)
plt.tight_layout()
plt.show()


## 3. Estadísticas de Canales y Formato de Tensores

Verificamos las dimensiones y el rango numérico de los batches entregados por el pipeline:


In [ ]:
print(f"Tensor Shape: {images.shape} -> (Batch Size, Canales, Alto, Ancho)")
print(f"Tipo de dato:  {images.dtype}")
print(f"Rango de valores normalizados: [{images.min():.2f}, {images.max():.2f}]")
